In [1]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import pandas as pd

def connect_to_database():
    DATABASE_URL = (
        "postgresql://postgres:1234"
        "@localhost:5432/google_and_spotify"
    )

    engine = create_engine(
        DATABASE_URL
    )
    return engine

In [3]:
engine = connect_to_database()

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.scalar())

1


In [4]:
df_visits = pd.read_sql(
    "SELECT * FROM fact_visit",
    engine
)
df_visits.head()

,visit_id,start_time,end_time,duration_minutes,candidate_id,probability,candidate_probability
0,1,2026-08-03 08:13:37.103000-03:00,2026-08-03 15:32:03.646000-03:00,438,ChIJiezqRFBoGZURscJvIoEdUrs,0.806735,0.599942
1,2,2026-08-03 08:14:22.196000-03:00,2026-08-03 15:32:03.646000-03:00,438,ChIJ_5DJhDloGZURrMFmY-T3yHM,0.806735,0.676057
2,3,2026-09-01 22:09:53.144000-03:00,2026-09-02 13:42:16.329000-03:00,932,ChIJb7yKt1FoGZUR3wOqGd0oc7s,0.813596,0.554897
3,4,2014-04-28 14:02:23.741000-03:00,2014-04-28 15:49:24.902000-03:00,107,ChIJ14C6tlFoGZURwuDID3ysPqY,0.770000,0.313333
4,5,2014-05-11 11:38:21.854000-03:00,2014-05-11 12:11:08.165000-03:00,33,ChIJ2e77KABpGZURe_fLtv-W7Tc,0.750000,0.291292


In [ ]:
import plotly.express as px

query = """
SELECT
    v.visit_id,
    v.start_time,
    v.end_time,
    v.candidate_id,
    c.name,
    c.latitude,
    c.longitude
FROM fact_visit AS v
LEFT JOIN dim_candidates AS c
    ON c.candidate_id = v.candidate_id
WHERE c.latitude IS NOT NULL
  AND c.longitude IS NOT NULL
ORDER BY v.start_time
"""

df_visits_map = pd.read_sql(query, engine)

df_visits_map["start_time"] = pd.to_datetime(
    df_visits_map["start_time"],
    utc=True,
)
df_visits_map["end_time"] = pd.to_datetime(
    df_visits_map["end_time"],
    utc=True,
)
df_visits_map["place_name"] = df_visits_map["name"].fillna(
    df_visits_map["candidate_id"]
)

fig = px.scatter_map(
    df_visits_map,
    lat="latitude",
    lon="longitude",
    hover_name="place_name",
    hover_data={
        "visit_id": True,
        "start_time": True,
        "end_time": True,
        "latitude": ":.6f",
        "longitude": ":.6f",
    },
    zoom=10,
    height=700,
    title="Visitas no mapa",
)
fig.update_traces(
    marker={"size": 9, "color": "#d73027"},
)
fig.update_layout(
    map_style="open-street-map",
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
)
fig.show()


ValueError: Tz-aware datetime.datetime cannot be converted to datetime64 unless utc=True, at position 45